# Private Soup T4 iteration
Generated configuration: use `python3 tools/kaggle/run.py run --mode unit`.
All commands run sequentially; logs, exit codes and evidence are saved in `/kaggle/working/out/`.

In [ ]:
MODE = "unit"
EXPECTED_SHA = None
DATASET = None
EXPERIMENT_ENV = {}
RUN_TAG = ""
AB_VARIANTS = None
AB_ROUNDS = 4
SESSION_ID = None


In [ ]:
import hashlib
import json
import os
import re
from pathlib import Path, PurePosixPath
import shutil
import stat
import subprocess
import sys
import traceback
import zipfile

OUT = Path('/kaggle/working/out')
REPO = Path('/kaggle/working/repo')
OUT.mkdir(parents=True, exist_ok=True)
status = {'mode': MODE, 'expected_commit': EXPECTED_SHA, 'dataset': DATASET,
          'commands': [], 'ok': False, 'env': EXPERIMENT_ENV, 'tag': RUN_TAG}

def save_status():
    (OUT / 'status.json').write_text(json.dumps(status, indent=2) + '\n')

def command(name, args, cwd=None, env=None):
    log = OUT / (name + '.log')
    entry = {'name': name, 'argv': args, 'exit_code': None}
    status['commands'].append(entry)
    save_status()
    try:
        with log.open('w') as output:
            proc = subprocess.run(args, cwd=cwd, env=env, stdout=output,
                                  stderr=subprocess.STDOUT, check=False)
        entry['exit_code'] = proc.returncode
    except OSError as exc:
        entry['exit_code'] = 127
        log.write_text(type(exc).__name__ + ': ' + str(exc))
    save_status()
    print(name, 'exit', entry['exit_code'])
    print('\n'.join(log.read_text(errors='replace').splitlines()[-12:]))
    return entry['exit_code']

def safe_path(name):
    path = PurePosixPath(name)
    if path.is_absolute() or '..' in path.parts or '.kaggle' in path.parts:
        raise ValueError('Unsafe archive path')
    return path

def copy_source():
    assert isinstance(EXPECTED_SHA, str) and len(EXPECTED_SHA) == 40, 'Use run.py to render this notebook'
    manifests = list(Path('/kaggle/input').rglob('source-manifest.json'))
    matching = []
    for path in manifests:
        manifest = json.loads(path.read_text())
        if manifest.get('commit') == EXPECTED_SHA:
            matching.append((path, manifest))
    assert len(matching) == 1, 'Attached dataset SHA does not match expected commit (or is ambiguous)'
    manifest_path, manifest = matching[0]
    base = manifest_path.parent
    shutil.rmtree(REPO, ignore_errors=True)
    REPO.mkdir(parents=True)
    archives = list(base.glob('source.zip'))
    if archives:
        archive = archives[0]
        assert hashlib.sha256(archive.read_bytes()).hexdigest() == manifest['archive_sha256'], 'ZIP hash mismatch'
        with zipfile.ZipFile(archive) as source:
            assert source.comment.decode('ascii').strip() == EXPECTED_SHA, 'git archive commit mismatch'
            for item in source.infolist():
                safe_path(item.filename)
                assert not stat.S_ISLNK(item.external_attr >> 16), 'Archive symlinks are unsupported'
            source.extractall(REPO)
    else:
        # Kaggle may auto-expand source.zip into source/ or the dataset root.
        roots = [base, base / 'source', base / 'repo']
        roots.extend(path.parent.parent for path in base.rglob('harness/run_task.py'))
        source_root = next((root for root in roots if all((root / name).is_file()
                           for name in manifest['files'])), None)
        assert source_root is not None, 'Cannot locate automatically extracted archive'
        for name in manifest['files']:
            safe_path(name)
            destination = REPO / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_root / name, destination)
    for name, expected in manifest['files'].items():
        safe_path(name)
        assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == expected, 'Source hash mismatch: ' + name
    shutil.copy2(manifest_path, OUT / 'source-manifest.json')
    status['verified_commit'] = EXPECTED_SHA
    save_status()
    print('Verified git archive HEAD:', EXPECTED_SHA)

def collect():
    # Preserve harness evidence, not downloaded datasets or adapter weights.
    for run_root in sorted(REPO.glob('runs*')):
        if not run_root.is_dir():
            continue
        for path in run_root.rglob('*'):
            if not path.is_file():
                continue
            if (path.name in {'result.json', 'checks.json', 'metrics.json', 'profile.json'}
                    or path.name.startswith('eval_') and path.suffix == '.json'
                    or path.suffix == '.log'):
                target = OUT / path.relative_to(REPO)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, target)
                if path.name == 'train.log':
                    target.with_name('train.tail.log').write_text(
                        '\n'.join(path.read_text(errors='replace').splitlines()[-100:]) + '\n')

save_status()
try:
    for key, value in EXPERIMENT_ENV.items():
        assert re.fullmatch(r'SOUP_[A-Za-z0-9_]+', key) and key != 'SOUP_APPLY_SOLUTION', 'Unsupported experiment key'
        assert isinstance(value, str) and '\0' not in value, 'Invalid environment value'
    os.environ.update(EXPERIMENT_ENV)
    rc = command('gpu', ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'])
    gpus = (OUT / 'gpu.log').read_text()
    assert rc == 0 and 'T4' in gpus, 'Switch the accelerator to GPU T4 x2'
    # Identical package arguments to templates/kaggle_notebook.ipynb.
    rc = command('pip', ['python', '-m', 'pip', 'install', '-q',
                         'soup-cli[train]==0.77.0', 'transformers==5.16.1',
                         'peft==0.20.0', 'trl==0.29.1', 'bitsandbytes==0.50.2',
                         'accelerate==1.14.0', 'pytest'])
    assert rc == 0, 'Pinned dependency installation failed; see pip.log'
    copy_source()
    if MODE == 'ab':
        rc = command('prepare_data', ['python', 'task/prepare_data.py'], cwd=REPO)
        assert rc == 0, 'Data preparation failed; see prepare_data.log'
        command('ab', ['python', 'tools/kaggle/registry.py', 'run-ab',
                       '--variants', AB_VARIANTS, '--rounds', str(AB_ROUNDS),
                       '--out', str(OUT / 'ab'), '--registry-root', str(OUT / 'experiments'),
                       '--commit', EXPECTED_SHA, '--session', SESSION_ID,
                       '--experiment', RUN_TAG or 'EXP-003'], cwd=REPO)
    elif MODE == 'bench':
        command('bench_kernels', ['python', 'tools/kaggle/bench_kernels.py', '--out', str(OUT), '--commit', EXPECTED_SHA], cwd=REPO)
    elif MODE == 'probe':
        shared = "import torch; torch.empty(1,device='cuda'); import transformers; import time; "
        soup = ("t=time.perf_counter(); import soup_cli.cli; "
                "from soup_cli.trainer.sft import SFTTrainerWrapper; "
                "from trl import SFTTrainer; "
                "from transformers.models.qwen2 import modeling_qwen2; "
                "print('soup_pre_model_s',time.perf_counter()-t)")
        with_solution = (shared + "import solution; t=time.perf_counter(); solution.apply(); "
                         "print('apply_s',time.perf_counter()-t); " + soup)
        command('probe_import_apply', ['python', '-X', 'importtime', '-c', with_solution], cwd=REPO)
        command('probe_import_baseline', ['python', '-X', 'importtime', '-c', shared + soup], cwd=REPO)
        command('probe_compile', ['python', 'tools/kaggle/probe.py', '--out', str(OUT)], cwd=REPO)
    else:
        rc = command('prepare_data', ['python', 'task/prepare_data.py'], cwd=REPO)
        assert rc == 0, 'Data preparation failed; see prepare_data.log'
        command('pytest_solution', ['python', '-m', 'pytest', 'solution', '-q', '-rs'], cwd=REPO)
        if MODE == 'unit':
            env = dict(os.environ, SOUP_APPLY_SOLUTION='1')
            command('pytest_soup', ['python', '-m', 'pytest', 'tests', '-q'], cwd=REPO, env=env)
        elif MODE == 'solution':
            command('harness', ['python', 'harness/run_task.py', '--only', 'solution',
                                '--out', 'runs_try', '--profile'], cwd=REPO)
        elif MODE in {'full', 'full2'}:
            args = ['python', 'harness/run_task.py']
            if MODE == 'full2':
                args += ['--repeats', '2']
            command('harness', args, cwd=REPO)
            # Run sequentially even after a harness failure to preserve diagnostics.
            command('checks', ['python', 'harness/check_rules.py'], cwd=REPO)
            command('score', ['python', 'harness/score.py', 'runs/result.json'], cwd=REPO)
        else:
            raise ValueError('Unsupported run mode')
    status['ok'] = all(item['exit_code'] == 0 for item in status['commands'])
except Exception as exc:
    # Save failure before leaving the cell; no harness error bypasses collection.
    status['fatal_error'] = type(exc).__name__ + ': ' + str(exc)
    (OUT / 'notebook-error.log').write_text(traceback.format_exc())
    print(status['fatal_error'])
finally:
    try:
        collect()
    except Exception as exc:
        status['collection_error'] = type(exc).__name__ + ': ' + str(exc)
        status['ok'] = False
    save_status()
    # A/B evidence is already under OUT; also prune interrupted run outputs.
    for run_dir in (OUT / 'ab' / 'runs').glob('*'):
        if run_dir.is_dir():
            for side in ('base_1', 'solution_1'):
                for name in ('adapter', 'triton_cache', 'inductor_cache'):
                    shutil.rmtree(run_dir / side / name, ignore_errors=True)
    # Avoid exporting model weights and prepared data as notebook output.
    if REPO.exists() and 'collection_error' not in status:
        shutil.rmtree(REPO)
    print('Saved logs and harness artifacts:', OUT)
    print(json.dumps(status, indent=2))
